In [1]:
# %pip install torch scikit-learn pandas numpy joblib

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import joblib
import math
from torch.utils.data import TensorDataset, DataLoader

In [3]:
# ============================================================
# Skoda Fabia Dataset
# Gefiltert auf Modell 'Fabia': 1.571 Proben
# Features: year, transmission, mileage, fuelType, tax, mpg, engineSize
# Ziel: price (GBP)
# Hinweis: Spalte 'model' wird nach dem Filter gedroppt --
#          konstante Spalten tragen keine Preisinformation.
# ============================================================

df_raw = pd.read_csv('../dataset/skoda.csv', skipinitialspace=True)
print(f'Rohdaten: {df_raw.shape[0]} Zeilen')

df = df_raw[df_raw['model'] == 'Fabia'].copy()  ### ANPASSEN
df = df.drop(columns=['model'])   # konstant -> kein Informationsgehalt
df = df.reset_index(drop=True)

print(f'Nach Filter Fabia: {df.shape[0]} Zeilen, {df.shape[1]} Spalten')
print(df.head())
print()
print(df.dtypes)
print()
print(df['price'].describe())

Rohdaten: 6267 Zeilen
Nach Filter Fabia: 1571 Zeilen, 8 Spalten
   year  price transmission  mileage fuelType  tax   mpg  engineSize
0  2012   4695       Manual    54631   Petrol  125  49.6         1.2
1  2012   4250       Manual    61000   Petrol  125  49.6         1.2
2  2017  11290       Manual    15282   Petrol  150  64.2         1.0
3  2019  13490       Manual     6000   Petrol  145  51.4         1.0
4  2017  11890    Automatic    10660   Petrol  150  60.1         1.2

year              int64
price             int64
transmission        str
mileage           int64
fuelType            str
tax               int64
mpg             float64
engineSize      float64
dtype: object

count     1571.000000
mean      9906.497136
std       2370.490798
min        995.000000
25%       8405.000000
50%       9790.000000
75%      11498.000000
max      17415.000000
Name: price, dtype: float64


In [4]:
# Features (X) und Ziel (y) vorbereiten
cat_cols = ['transmission', 'fuelType']
# Spaltenreihenfolge nach drop('price'):
# year, transmission, mileage, fuelType, tax, mpg, engineSize

# OrdinalEncoder auf Gesamtdaten fitten (pragmatisch):
# Wuerde man ihn nur auf Train fitten, wuerfen unbekannte Kategorien
# im Test-Split einen ValueError.
enc = OrdinalEncoder()
df[cat_cols] = enc.fit_transform(df[cat_cols])

X = df.drop('price', axis=1).values      # (1571, 7)  -- noch unskaliert
y = df['price'].values.reshape(-1, 1)    # (1571, 1)  -- noch unskaliert

print(f'X-Shape: {X.shape} | y-Shape: {y.shape}')
print(f'Features: {list(df.drop("price", axis=1).columns)}')

X-Shape: (1571, 7) | y-Shape: (1571, 1)
Features: ['year', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize']


In [5]:
# -- Erst splitten, DANN skalieren -> kein Data Leakage --
# Wuerde man den Scaler vorher auf allen Daten fitten, fliesst die
# Verteilung (Mittelwert, Std) von Val/Test in den Scaler ein.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=0
)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

# Scaler NUR auf Trainingsdaten fitten
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)  # fit + transform
X_val   = scaler_X.transform(X_val)         # nur transform
X_test  = scaler_X.transform(X_test)         # nur transform

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)   # fit + transform
y_val   = scaler_y.transform(y_val)          # nur transform
y_test  = scaler_y.transform(y_test)          # nur transform

# Umwandeln in PyTorch-Tensoren
X_train_t = torch.from_numpy(X_train).float()
X_val_t   = torch.from_numpy(X_val).float()
X_test_t  = torch.from_numpy(X_test).float()
y_train_t = torch.from_numpy(y_train).float()
y_val_t   = torch.from_numpy(y_val).float()
y_test_t  = torch.from_numpy(y_test).float()

# DataLoader erstellen
train_dataset = TensorDataset(X_train_t, y_train_t)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

Train: 1004 | Val: 252 | Test: 315


In [6]:
# ============================================================
# Neuronales Netz (Regression)
# Experimentier-Stellschrauben:
#   - Anzahl Neuronen in den Hidden Layers
#   - Aktivierungsfunktion (ReLU, Tanh, LeakyReLU, GELU ...)
#   - Dropout-Rate
#   - Optimizer (SGD, Adam, AdamW, RMSprop ...)
#   - Batch-Size (oben)
#   - Lernrate
#   - Loss-Funktion (MSELoss, HuberLoss, L1Loss)
# ============================================================

net = nn.Sequential(
    nn.Linear(7, 64),      # 7 Features -> Hidden Layer 1
    nn.ReLU(),
    nn.Dropout(0.2),       # <-- hier Dropout-Rate variieren (0.0 bis 0.5)
    nn.Linear(64, 32),     # Hidden Layer 2
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 1)       # 1 Ausgabe: Preis (skaliert)
)

# HuberLoss: robuster gegen Ausreisser als MSELoss
criterion = nn.MSELoss()        # nn.HuberLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.001)

num_epochs = 100
net.train()
for epoch in range(num_epochs):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = net(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    # Validierung nach jeder Epoche
    net.eval()
    with torch.no_grad():
        val_out  = net(X_val_t)
        val_loss = criterion(val_out, y_val_t)
        val_rmse = math.sqrt(mean_squared_error(
            scaler_y.inverse_transform(y_val_t.numpy()),
            scaler_y.inverse_transform(val_out.numpy())
        ))
    net.train()

    if epoch % 10 == 0 or epoch == num_epochs - 1:
        print(f'Epoch {epoch:3d} | Train-Loss: {loss.item():.4f} | Val-Loss: {val_loss.item():.4f} | Val-RMSE: {val_rmse:,.0f} GBP')

Epoch   0 | Train-Loss: 0.6178 | Val-Loss: 0.5750 | Val-RMSE: 1,776 GBP
Epoch  10 | Train-Loss: 0.2453 | Val-Loss: 0.2011 | Val-RMSE: 1,051 GBP
Epoch  20 | Train-Loss: 0.1819 | Val-Loss: 0.1749 | Val-RMSE: 980 GBP
Epoch  30 | Train-Loss: 0.1602 | Val-Loss: 0.1625 | Val-RMSE: 944 GBP
Epoch  40 | Train-Loss: 0.3817 | Val-Loss: 0.1636 | Val-RMSE: 948 GBP
Epoch  50 | Train-Loss: 0.3001 | Val-Loss: 0.1519 | Val-RMSE: 913 GBP
Epoch  60 | Train-Loss: 0.2069 | Val-Loss: 0.1650 | Val-RMSE: 952 GBP
Epoch  70 | Train-Loss: 0.1998 | Val-Loss: 0.1455 | Val-RMSE: 894 GBP
Epoch  80 | Train-Loss: 0.1400 | Val-Loss: 0.1494 | Val-RMSE: 906 GBP
Epoch  90 | Train-Loss: 0.0982 | Val-Loss: 0.1392 | Val-RMSE: 874 GBP
Epoch  99 | Train-Loss: 0.1679 | Val-Loss: 0.1547 | Val-RMSE: 921 GBP


In [7]:
# Evaluation auf Testdaten
net.eval()
with torch.no_grad():
    test_out = net(X_test_t)

y_test_orig = scaler_y.inverse_transform(y_test_t.numpy())
y_pred_orig = scaler_y.inverse_transform(test_out.numpy())

rmse = math.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
mae  = mean_absolute_error(y_test_orig, y_pred_orig)
r2   = r2_score(y_test_orig, y_pred_orig)

print(f'Test-RMSE: {rmse:>10,.0f} GBP')
print(f'Test-MAE:  {mae:>10,.0f} GBP')
print(f'Test-R2:   {r2:>10.4f}')

Test-RMSE:      1,005 GBP
Test-MAE:         796 GBP
Test-R2:       0.8273


In [8]:
# Modell und Hilfsobjekte speichern
torch.save(net.state_dict(), 'fabia_net.pth')
print("Trainiertes Netz abgespeichert als 'fabia_net.pth'")
joblib.dump(enc,      'fabia_ordinal_encoder.pkl')
joblib.dump(scaler_X, 'fabia_scaler_X.pkl')
joblib.dump(scaler_y, 'fabia_scaler_y.pkl')
print('Encoder und Scaler gespeichert.')

Trainiertes Netz abgespeichert als 'fabia_net.pth'
Encoder und Scaler gespeichert.


In [9]:
# Wiederladen des trainierten Netzes
import torch
import torch.nn as nn
import joblib

enc      = joblib.load('fabia_ordinal_encoder.pkl')
scaler_X = joblib.load('fabia_scaler_X.pkl')
scaler_y = joblib.load('fabia_scaler_y.pkl')

# Architektur muss identisch sein!
net = nn.Sequential(
    nn.Linear(7, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 1)
)
net.load_state_dict(torch.load('fabia_net.pth', map_location=torch.device('cpu')))

for k, v in net.named_parameters():
    print(k, v.shape)

net.eval()

0.weight torch.Size([64, 7])
0.bias torch.Size([64])
3.weight torch.Size([32, 64])
3.bias torch.Size([32])
6.weight torch.Size([1, 32])
6.bias torch.Size([1])


Sequential(
  (0): Linear(in_features=7, out_features=64, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=64, out_features=32, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.2, inplace=False)
  (6): Linear(in_features=32, out_features=1, bias=True)
)

In [11]:
# Vorhersage mit dem trainierten Netz
# Modell ist fix: Fabia
# Spaltenreihenfolge: year, transmission, mileage, fuelType, tax, mpg, engineSize
import numpy as np
import pandas as pd

cat_cols = ['transmission', 'fuelType']

print('Gueltige Werte fuer kategoriale Features:')
for col, cats in zip(cat_cols, enc.categories_):
    print(f'  {col}: {list(cats)}')

print('\nFabia-Daten eingeben:')
year         = float(input('year         : '))
transmission = input('transmission : ')
mileage      = float(input('mileage      : '))
fuelType     = input('fuelType     : ')
tax          = float(input('tax          : '))
mpg          = float(input('mpg          : '))
engineSize   = float(input('engineSize   : '))

# Eingabe als DataFrame in der gleichen Spaltenreihenfolge wie beim Training
new_df = pd.DataFrame([{
    'year':         year,
    'transmission': transmission,
    'mileage':      mileage,
    'fuelType':     fuelType,
    'tax':          tax,
    'mpg':          mpg,
    'engineSize':   engineSize
}])

new_df[cat_cols] = enc.transform(new_df[cat_cols])
row_scaled = scaler_X.transform(new_df.values.astype(float))
inputs = torch.tensor(row_scaled, dtype=torch.float32)

with torch.no_grad():
    output = net(inputs)

price = scaler_y.inverse_transform(output.numpy())[0][0]
print(f'\nVorhergesagter Preis (Fabia): {price:,.0f} GBP')

Gueltige Werte fuer kategoriale Features:
  transmission: ['Automatic', 'Manual', 'Semi-Auto']
  fuelType: ['Diesel', 'Other', 'Petrol']

Fabia-Daten eingeben:

Vorhergesagter Preis (Fabia): 11,255 GBP
